In [21]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset
from datasets import DatasetDict
from transformers import pipeline
from transformers import DataCollatorForSeq2Seq
from transformers import Seq2SeqTrainingArguments
from transformers import Seq2SeqTrainer
import evaluate

In [5]:
raw_datasets= load_dataset("ngia/translation-en-fr")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/596 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

data/train-00000-of-00030.parquet:   0%|          | 0.00/316M [00:00<?, ?B/s]

...

data/train-00029-of-00030.parquet:   0%|          | 0.00/316M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/95.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/40252491 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/406591 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/30 [00:00<?, ?it/s]

In [6]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['english_src', 'french_tgt'],
        num_rows: 40252491
    })
    test: Dataset({
        features: ['english_src', 'french_tgt'],
        num_rows: 406591
    })
})

In [7]:
raw_datasets['train'][0]

{'english_src': 'Mr President, the world gets smaller every day, and this is now so obviously the case that the globalization of the economy is something which everyone accepts as normal.',
 'french_tgt': "Monsieur le Président, le monde devient de plus en plus petit, ce constat est d'autant plus évident que la globalisation ou mondialisation de l'économie est un phénomène que tout le monde trouve naturel."}

In [8]:
small_datasets = DatasetDict({
    "train": raw_datasets["train"].shuffle(seed=42).select(range(10000)),
    "test": raw_datasets["test"].shuffle(seed=42).select(range(2000)),
})
small_datasets

DatasetDict({
    train: Dataset({
        features: ['english_src', 'french_tgt'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['english_src', 'french_tgt'],
        num_rows: 2000
    })
})

In [9]:
model_checkpoint = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, return_tensors="pt")
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/301M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/301M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

In [10]:
def preprocess_function(examples):
    inputs = examples["english_src"]
    targets = examples["french_tgt"]

    model_inputs = tokenizer(
        inputs,
        text_target=targets,
        max_length=128,
        truncation=True,
    )
    return model_inputs

tokenized_datasets= small_datasets.map(preprocess_function, batched=True, remove_columns=small_datasets["train"].column_names)
tokenized_datasets

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2000
    })
})

In [11]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [12]:
batch = data_collator([tokenized_datasets["train"][i] for i in range(1, 3)])
batch.keys()

KeysView({'input_ids': tensor([[ 1438,  4871,    37,  8412, 23803,  1437,   365,    37, 13598,  2786,
         48224,    48,    77,  6087,   160,   292,    32,   494,    30,     4,
         24930,  1736,   106,  2115,     3,     0, 59513, 59513, 59513, 59513,
         59513],
        [  660,  3950,    52,    73, 10520,   761,    46, 19654,    48,   218,
            52,  4465,   420,  1942,    21,  1876,  1130,    24, 42343,  1105,
           232,   544,    21, 11267,     9,     6, 19654,    71,   967,   102,
             0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1]]), 'labels': tensor([[ 9593,    22,   230,    37,  8412,  6485, 20596,    78, 13598,  2786,
          2916,  1349,    43,    38,  1780,   335,   230,    43,  2200,    36,
            19,   414,  1232,    13, 33584, 42505,   10

In [16]:
metric = evaluate.load("sacrebleu")

In [17]:
args= Seq2SeqTrainingArguments(
    "test-translation",
    # evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    # save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
    push_to_hub=False,
)

trainer= Seq2SeqTrainer(
    model, 
    args, 
    train_dataset=tokenized_datasets["train"], 
    eval_dataset=tokenized_datasets["test"], 
    data_collator=data_collator, 
    # tokenizer= tokenizer, 
    # compute_metrics=metric.compute
)
trainer.train()

Step,Training Loss
500,1.392271
1000,1.209082
1500,1.170081


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1875, training_loss=1.2263154134114584, metrics={'train_runtime': 834.0172, 'train_samples_per_second': 35.97, 'train_steps_per_second': 2.248, 'total_flos': 636263835107328.0, 'train_loss': 1.2263154134114584, 'epoch': 3.0})

In [22]:
device= torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

MarianMTModel(
  (model): MarianModel(
    (shared): Embedding(59514, 512, padding_idx=59513)
    (encoder): MarianEncoder(
      (embed_tokens): Embedding(59514, 512, padding_idx=59513)
      (embed_positions): MarianSinusoidalPositionalEmbedding(512, 512)
      (layers): ModuleList(
        (0-5): 6 x MarianEncoderLayer(
          (self_attn): MarianAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation_fn): SiLU()
          (fc1): Linear(in_features=512, out_features=2048, bias=True)
          (fc2): Linear(in_features=2048, out_features=512, bias=True)
          (final_layer_norm): LayerNorm((512,), eps=1e-05

In [ ]:
text="He is a dancer."
inputs= tokenizer(text, return_tensors="pt").to(device)
outputs= model.generate(**inputs).to(device)
print(tokenizer.decode(outputs[0]))

<pad> C'est un danseur.</s>


In [27]:
model.save_pretrained("./en-fr_model")
tokenizer.save_pretrained("./en-fr_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./en-fr_model/tokenizer_config.json',
 './en-fr_model/vocab.json',
 './en-fr_model/source.spm',
 './en-fr_model/target.spm',
 './en-fr_model/added_tokens.json')

In [28]:
!zip -r en-fr_model.zip en-fr_model

  adding: en-fr_model/ (stored 0%)
  adding: en-fr_model/config.json (deflated 63%)
  adding: en-fr_model/model.safetensors (deflated 7%)
  adding: en-fr_model/tokenizer_config.json (deflated 67%)
  adding: en-fr_model/source.spm (deflated 49%)
  adding: en-fr_model/target.spm (deflated 50%)
  adding: en-fr_model/vocab.json (deflated 70%)
  adding: en-fr_model/generation_config.json (deflated 43%)


In [31]:
from google.colab import files
files.download("en-fr_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>